# Lab 5.1 &mdash; The Supervisor Is a Router

**Level:** Intermediate &rarr; Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 2 &middot; Module 5 &mdash; Multi-Agent Collaboration &amp; Orchestration**

### What you'll do
- Build a rule-based supervisor, and measure its routing accuracy honestly
- Read the confusion table &mdash; and notice where every unrecognised request piles up
- Price a misroute: the wasted tokens are everything downstream of the mistake
- Wire the supervisor into a real <code>StateGraph</code> with <code>add_conditional_edges</code>

> **How this lab works.** You write real LangGraph code. Fill every `BLANK`, then run the
> **Self-check** cell under each section &mdash; those assert on the *objects you built*
> (a compiled `StateGraph`, a declared reducer, a checkpointed interrupt), so they are
> deterministic and never depend on the model. Cells marked **Run it for real** put your work
> in front of the sandbox model; that is the part worth watching. The score line is feedback,
> not a grade.

> **Module 3's graph, with several workers in it.** A supervisor picks a specialist the
> way an agent picks a tool &mdash; so it is a conditional edge, and it has an accuracy.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-5-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def _is_todo(exc: BaseException) -> bool:
    """Is this exception really an unfilled blank?

    LangGraph runs your nodes inside tasks, so the NameError from an unfilled BLANK can
    arrive wrapped. Walk the cause chain before calling anything a failure -- telling you
    your answer is wrong when you have not written one yet is the worst thing a lab does.
    """
    seen = set()
    while exc is not None and id(exc) not in seen:
        if isinstance(exc, NameError):
            return True
        seen.add(id(exc))
        exc = exc.__cause__ or exc.__context__
    return False

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except Exception as exc:
        if _is_todo(exc):
            print(f"[TODO] {name}")
            _results.append(None)
            return
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except Exception as exc:
        if not _is_todo(exc):
            raise
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Off is the default here because the "Run it for real" cells make many small calls.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 5 labs -- the same payment exceptions, now worked
# by several agents at once, and finally priced against the single agent from Day 1.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- the specialists, as LangGraph nodes
# Each one takes the graph state and returns a PARTIAL state -- exactly the node shape from
# Module 3 -- and reports what it spent. They are deterministic, so a graph's structure AND
# its cost can be asserted offline and exactly. The "Run it for real" cells put the sandbox
# model behind the same interface.

SANCTIONS_WATCH = {"NORTHWIND"}

COST = {"supervisor": 120, "ledger": 380, "policy": 420, "sanctions": 90, "writer": 610}


def agent_ledger(state: dict) -> dict:
    """Read the payment named in the state."""
    ref = state.get("ref")
    record = LEDGER.get(ref)
    if record is None:
        return {"problems": [f"no payment on file with reference {ref!r}"],
                "tokens": COST["ledger"]}
    return {"facts": {"ref": ref, **record},
            "findings": [{"by": "ledger", "source": "ledger",
                          "claim": f"{ref} is {record['status']} "
                                   f"for {record['amount']:,.2f} {record['ccy']}"}],
            "tokens": COST["ledger"]}


def agent_policy(state: dict) -> dict:
    """Say what the operating policy is for whatever went wrong."""
    code = (state.get("facts") or {}).get("reason_code")
    if code is None:
        return {"problems": ["policy ran before the reason code existed"],
                "tokens": COST["policy"]}
    return {"findings": [{"by": "policy", "source": "policy",
                          "claim": POLICY.get(code, f"no policy on file for {code}")}],
            "needs_human": code in NEEDS_HUMAN,
            "tokens": COST["policy"]}


def agent_sanctions(state: dict) -> dict:
    """A set-membership test. No model needed, and none used -- note the cost column."""
    counterparty = (state.get("facts") or {}).get("counterparty")
    listed = counterparty in SANCTIONS_WATCH
    return {"findings": [{"by": "sanctions", "source": "watchlist",
                          "claim": f"{counterparty} is "
                                   f"{'ON the watchlist' if listed else 'not on the watchlist'}"}],
            "blocked": listed,
            "tokens": COST["sanctions"]}


def agent_writer(state: dict) -> dict:
    """Turn whatever findings arrived into one recommendation."""
    findings = state.get("findings") or []
    if (state.get("facts") or {}).get("status") == "settled":
        action = "no action"                      # nothing to release; it already went
    elif state.get("blocked") or state.get("needs_human"):
        action = "hold for a human"
    else:
        action = "release"
    return {"recommendation": action,
            "rationale": [f["claim"] for f in findings],
            "tokens": COST["writer"]}


AGENTS = {"ledger": agent_ledger, "policy": agent_policy,
          "sanctions": agent_sanctions, "writer": agent_writer}
print("specialists:", ", ".join(AGENTS))

## Concept

A supervisor decides which specialist handles a request. In LangGraph that is one thing: a
**conditional edge** out of a supervisor node. `add_conditional_edges(node, fn, path_map)` calls
`fn(state)`, which returns a key of `path_map`, and the graph goes there.

Which makes the supervisor a classifier with a known correct answer &mdash; so it has an accuracy,
and almost nobody measures it.

It is worth measuring because a misroute is the most expensive mistake in the graph: everything
spent downstream of it answered the wrong question. The supervisor's own call is the cheapest one
in the system, so &ldquo;save money on the router&rdquo; is usually a bad trade.

## Section 1 &mdash; A rule-based supervisor

Keywords, in order, with a fallback. Unglamorous, free, instant, and identical every time &mdash;
and right far more often than people expect.

The mechanics are written out. The decision left to you is the **fallback**, because every
request the table does not recognise ends up there, which makes that one line the router's
entire failure mode.

In [ ]:
SPECIALISTS = ("ledger", "policy", "sanctions", "writer")

# Order matters: the most specific group goes first.
ROUTING_KEYWORDS = [
    ("sanctions", ("sanction", "embargo", "screening")),
    ("policy",    ("policy", "runbook", "rule", "limit", "breach", "allowed", "permitted")),
    ("writer",    ("draft", "write", "note", "letter", "summar")),
    ("ledger",    ("status", "amount", "record", "look up", "reference", "pull")),
]

def route_by_rule(request: str) -> str:
    """The first keyword group that matches wins."""
    low = (request or "").lower()
    for specialist, keywords in ROUTING_KEYWORDS:
        if any(k in low for k in keywords):
            return specialist
    # Nothing matched. The ledger read is the only step that needs no other findings first,
    # so an unrecognised request is at least started rather than answered wrongly.
    return "ledger"

In [ ]:
# --- Self-check: Section 1
check("an explicit sanctions request routes to sanctions",
      lambda: route_by_rule("Run the embargo check on ZENITH.") == "sanctions")
check("a policy question routes to policy",
      lambda: route_by_rule("What is the runbook for an INVALID_IBAN return?") == "policy")
check("a lookup routes to the ledger",
      lambda: route_by_rule("What is the status of PMT-1001?") == "ledger")
check("a drafting request routes to the writer",
      lambda: route_by_rule("Draft the customer note for PMT-1002.") == "writer")
check("the fallback is a specialist that actually exists",
      lambda: route_by_rule("zzzz nothing matches here zzzz") in SPECIALISTS,
      "a conditional edge that returns a key the path map does not have is a runtime error")
check("the fallback is the one step that needs no prior findings",
      lambda: route_by_rule("zzzz nothing matches here zzzz") == "ledger",
      "policy needs a reason code, sanctions needs a counterparty, the writer needs findings")
check("every rule points at a specialist that exists",
      lambda: all(s in SPECIALISTS for s, _ in ROUTING_KEYWORDS))

## Section 2 &mdash; Measure it

Fifteen requests with a known correct specialist. Four of them state their intent only by
implication &mdash; no keyword names it &mdash; because those are the cases a rule table cannot reach and
the reason anyone reaches for a model. This whole section is given: nothing here is a design
decision, it is the harness.

In [ ]:
ROUTE_EVAL = [
    ("Is PMT-1005 clear of sanctions screening?",                  "sanctions"),
    ("Run the embargo check on ZENITH.",                           "sanctions"),
    ("PMT-1003 breached the limit -- what does policy say?",       "policy"),
    ("What is the runbook for an INVALID_IBAN return?",            "policy"),
    ("Are we allowed to retry this one automatically?",            "policy"),
    ("What is the status of PMT-1001?",                            "ledger"),
    ("Look up the amount on reference PMT-1004.",                  "ledger"),
    ("Pull the record for PMT-1002.",                              "ledger"),
    ("Draft the customer note for PMT-1002.",                      "writer"),
    ("Write up the case summary for the file.",                    "writer"),
    ("Summarise why this payment is held and what happens next.",  "writer"),
    # the four whose intent is implied, not stated
    ("Who is the counterparty on PMT-1003, and is that name a problem?",       "sanctions"),
    ("Is there anything about ZENITH we should worry about before releasing?", "sanctions"),
    ("This one has been sitting for three days. What are we supposed to do?",  "policy"),
    ("Tell the client what happened and why.",                                 "writer"),
]

def selections(router) -> dict:
    """{request: chosen specialist} for the whole eval set."""
    return {request: router(request) for request, _ in ROUTE_EVAL}


def accuracy(sel: dict) -> float:
    """Fraction routed to the expected specialist. No selection counts as wrong."""
    return sum(1 for r, expected in ROUTE_EVAL if sel.get(r) == expected) / len(ROUTE_EVAL)


def confusion(sel: dict) -> dict:
    """{(expected, chosen): count} over the misses -- the pairs whose boundaries overlap."""
    out = {}
    for request, expected in ROUTE_EVAL:
        chosen = sel.get(request)
        if chosen != expected:
            out[(expected, chosen)] = out.get((expected, chosen), 0) + 1
    return out


def _report():
    sel = selections(route_by_rule)
    print(f"rule-based supervisor: {accuracy(sel):.0%} on {len(ROUTE_EVAL)} requests\n")
    for (expected, chosen), n in sorted(confusion(sel).items(), key=lambda kv: -kv[1]):
        print(f"  {n}x  should have been {expected:10} -> went to {chosen}")
guard(_report)

In [ ]:
# --- Self-check: Section 2
_rule = None
def rule_selections():
    global _rule
    if _rule is None:
        _rule = selections(route_by_rule)
    return _rule

check("the eval set covers every specialist",
      lambda: {e for _, e in ROUTE_EVAL} == set(SPECIALISTS))
check("it contains requests whose intent is only implied",
      lambda: sum(1 for r, _ in ROUTE_EVAL
                  if not any(k in r.lower() for _, ks in ROUTING_KEYWORDS for k in ks)) >= 4,
      "an eval set of keyword-shaped requests measures the keywords, not the routing")
check("the rule supervisor gets most of it right",
      lambda: accuracy(rule_selections()) > 0.6)
check("but not all of it -- there is headroom to argue about",
      lambda: accuracy(rule_selections()) < 1.0)
check("every miss lands on the FALLBACK, not on a random specialist",
      lambda: {chosen for _, chosen in confusion(rule_selections())} == {"ledger"},
      "a rule router's failure mode is its fallback -- that is where unrecognised intent piles up")
check("so the confusion table names one problem, not four",
      lambda: len({chosen for _, chosen in confusion(rule_selections())}) == 1)

## Section 3 &mdash; What a misroute costs

The supervisor's own call is the cheapest thing in the graph. The specialist it wakes up is not.
Price the mistake and the argument about which model to route with settles itself.

In [ ]:
# COST came with the specialists. The supervisor is charged too -- routing is not free.

def cost_of(chosen: str) -> int:
    """Tokens for one routing decision plus the specialist it woke up."""
    return COST["supervisor"] + COST.get(chosen, 0)


def wasted_tokens(sel: dict) -> int:
    """Tokens spent answering the wrong question."""
    total = 0
    for request, expected in ROUTE_EVAL:
        chosen = sel.get(request)
        if chosen and chosen != expected:
            # The whole hop is wasted: you paid to choose wrongly and then paid the wrong
            # specialist. Charging only the specialist flatters the router that caused it.
            total += cost_of(chosen)
    return total


def spent_tokens(sel: dict) -> int:
    """Everything the run spent, right or wrong."""
    return sum(cost_of(sel[r]) for r, _ in ROUTE_EVAL if sel.get(r))

In [ ]:
# --- Self-check: Section 3
_perfect = {r: e for r, e in ROUTE_EVAL}

check("a perfect router wastes nothing",
      lambda: wasted_tokens(_perfect) == 0)
check("the waste is the supervisor call plus the specialist it woke up",
      lambda: wasted_tokens({**_perfect,
                             "Tell the client what happened and why.": "ledger"})
              == COST["supervisor"] + COST["ledger"],
      "the routing call is part of the mistake, not a sunk cost -- you would not have made it")
check("misrouting to the writer costs more than misrouting to sanctions",
      lambda: cost_of("writer") > cost_of("sanctions"),
      "the cost of a mistake depends on which specialist you woke up, not on the mistake")
check("the rule router wastes a real fraction of what it spends",
      lambda: 0 < wasted_tokens(rule_selections()) < spent_tokens(rule_selections()))
check("cheapening the supervisor cannot recover that waste",
      lambda: wasted_tokens(rule_selections()) > COST["supervisor"] * len(ROUTE_EVAL),
      "even a FREE supervisor would not save what the misroutes already cost -- that is the whole point")

def _price():
    sel = rule_selections()
    spent, wasted = spent_tokens(sel), wasted_tokens(sel)
    print(f"  spent   {spent:>6} tokens")
    print(f"  wasted  {wasted:>6} tokens  ({wasted / spent:.0%} of the bill)")
    print(f"  the supervisor's own calls were only {COST['supervisor'] * len(ROUTE_EVAL)} of that")
guard(_price)

## Section 4 &mdash; The supervisor as a real conditional edge

Everything above was a function returning a string. Now make it a graph.

`add_conditional_edges(source, fn, path_map)` needs two different things and people mix them up:

| | |
|---|---|
| `fn` | a function of **state** that returns a **key** |
| `path_map` | `{key: node name}` &mdash; which node each key means |

`route_by_rule` is *not* `fn`: it takes a request string, not a state. Write the one-line adapter
that is.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END

class RouterState(TypedDict):
    request: str                              # what the user asked for
    ref: str                                  # the payment the request is about
    route: str | None                         # the supervisor's decision, written into state
    facts: dict | None
    findings: Annotated[list, add]            # append: every node's findings survive
    problems: Annotated[list, add]
    tokens: Annotated[int, add]               # add: the bill is the sum, not the last write
    blocked: bool
    needs_human: bool
    recommendation: str | None
    rationale: list


def supervisor_node(state: RouterState) -> dict:
    """The supervisor is a node like any other. It decides, and it charges for deciding."""
    return {"route": route_by_rule(state["request"]), "tokens": COST["supervisor"]}


def choose_specialist(state: RouterState) -> str:
    """The adapter: takes STATE, returns a KEY of the path map."""
    return state["route"]


def build_router_graph():
    g = StateGraph(RouterState)
    g.add_node("supervisor", supervisor_node)
    for name in SPECIALISTS:
        g.add_node(name, AGENTS[name])        # the specialists, unchanged, as nodes
    g.add_edge(START, "supervisor")
    g.add_conditional_edges("supervisor", choose_specialist, {n: n for n in SPECIALISTS})
    for name in SPECIALISTS:
        g.add_edge(name, END)
    return g.compile()


def fresh_router_state(request: str, ref: str = "PMT-1005") -> dict:
    return {"request": request, "ref": ref, "route": None, "facts": None,
            "findings": [], "problems": [], "tokens": 0, "blocked": False,
            "needs_human": False, "recommendation": None, "rationale": []}

In [ ]:
# --- Self-check: Section 4   (a REAL compiled graph, running -- still no model)
def _via_graph(request: str) -> dict:
    return build_router_graph().invoke(fresh_router_state(request))

check("the supervisor graph compiles",
      lambda: build_router_graph() is not None)
check("the supervisor's decision is written into state, not hidden in control flow",
      lambda: _via_graph("What is the status of PMT-1005?")["route"] == "ledger",
      "a routing decision you cannot read back is one you cannot audit")
check("and the ledger node really ran",
      lambda: _via_graph("What is the status of PMT-1005?")["facts"]["ref"] == "PMT-1005")
check("a sanctions request reaches the sanctions specialist",
      lambda: _via_graph("Run the embargo check on ZENITH.")["route"] == "sanctions")
check("exactly one specialist runs per request",
      lambda: len(_via_graph("Run the embargo check on ZENITH.")["findings"]) == 1,
      "a conditional edge picks ONE path; fanning out to all four is a different design "
      "and Lab 5.2 costs it")
check("but that specialist had no case facts to work with",
      lambda: _via_graph("Run the embargo check on ZENITH.")["blocked"] is False,
      "nothing read the payment first, so the screen had no counterparty -- ordering is Lab 5.2")
check("the run is charged for the routing decision as well as the specialist",
      lambda: _via_graph("What is the status of PMT-1005?")["tokens"]
              == COST["supervisor"] + COST["ledger"],
      "that total is the Annotated[int, add] reducer on `tokens` doing the adding")
check("the graph agrees with the function it was built from",
      lambda: all(_via_graph(r)["route"] == route_by_rule(r) for r, _ in ROUTE_EVAL[:4]))

def _trace():
    for chunk in build_router_graph().stream(
            fresh_router_state("Summarise why PMT-1005 is held and what happens next.")):
        for node, update in chunk.items():
            print(f"  {node:12} -> {list(update)}")
guard(_trace)

## Run it for real &mdash; the model behind the same interface

Same eval set, same metric, same confusion table, same conditional edge. The only thing that
changes is the function inside `route_with_model`.

In [ ]:
ROUTE_SYSTEM = ("You route one operations request to exactly one specialist. "
                "Reply with the specialist's name alone -- no punctuation, no explanation.")

SPECIALIST_DESCRIPTIONS = {
    "ledger":    "Reads one payment record: status, amount, counterparty, reason code.",
    "policy":    "Says what the operating policy or runbook requires for a failure reason.",
    "sanctions": "Screens a counterparty name against the watchlist.",
    "writer":    "Turns findings into a summary or a customer-facing note.",
}

def route_with_model(request: str) -> str:
    """Ask the model to pick a specialist. Anything unrecognised falls back to the rules."""
    listing = "\n".join(f"- {n}: {d}" for n, d in SPECIALIST_DESCRIPTIONS.items())
    reply = ask(f"Specialists:\n{listing}\n\nRequest: {request}\n\nSpecialist:",
                system=ROUTE_SYSTEM)
    word = (reply or "").strip().strip("`.\"' ").split()
    return word[0] if word and word[0] in SPECIALIST_DESCRIPTIONS else route_by_rule(request)


if llm_ready():
    def _compare():
        rule = rule_selections()
        model = selections(route_with_model)
        print(f"{'router':16}{'accuracy':>10}{'wasted tokens':>16}")
        print("-" * 44)
        print(f"{'rule-based':16}{accuracy(rule):>9.0%}{wasted_tokens(rule):>16}")
        print(f"{'model':16}{accuracy(model):>9.0%}{wasted_tokens(model):>16}")
        print()
        for (expected, chosen), n in sorted(confusion(model).items(), key=lambda kv: -kv[1]):
            print(f"  model: {n}x  {expected} -> {chosen}")
    guard(_compare)

### Read it

Three things to look at, and the second is the one that decides your design:

1. **Did the model beat the rule table?** If not, the rules are free and deterministic, and you
   have your answer.
2. **Where did the model's misses land?** The rule router's misses all pile up on the fallback,
   which is one problem you can name. If the model's misses are scattered across four specialists,
   that is four overlapping descriptions &mdash; and Module 4 told you how to fix each one.
3. **Run it twice.** If the same request routes differently on the second run, you have met
   Module 7's opening problem a day early.

The usual production answer is neither: rules for the requests you can name, a model only for the
ones that fall through &mdash; which is exactly what `route_with_model`'s fallback line already does.
And note that swapping routers changed **no graph code at all**: the conditional edge does not
care where the key came from.

In [ ]:
score()

## Your turn

1. Make the hybrid explicit: try the rules, and call the model *only* when nothing matched.
   Measure its accuracy and its cost, and decide whether the saving is worth the second code path.
2. `route_by_rule` returns the first match, so a request mentioning both a policy and a draft goes
   to policy. Have `choose_specialist` return a **list** of keys instead &mdash; `add_conditional_edges`
   accepts that and dispatches to all of them. What have you just committed to paying?
3. Add a fifth key to the path map: `clarify` &mdash; requests where the honest answer is a question
   back to the user. What does that do to your accuracy, and is the drop real?